# Week 1 · Day 3 — Lab 1
## Core Structures: Series, DataFrame & Index

> **AI Engineering Academy** · Gamut Technology Services · pandas 3.x

pandas has exactly three objects you need to internalize before anything else:
the **Series** (a one-dimensional labeled array — every DataFrame column is one),
the **DataFrame** (a dict of aligned Series sharing one **Index**), and the
**Index** itself (the row-label axis that powers fast lookup and automatic
alignment). Get these right and the rest of pandas is combinations of them.

This lab is deliberately about the *mental model*. You'll build these objects by
hand, read their dtypes (including pandas 3.x's new automatic `str` dtype), and
see the Index do the alignment work that makes pandas more than a spreadsheet.

### Learning objectives
1. Construct a `Series` from a list and from a dict, and read its `dtype`, `index`, and `name`.
2. Construct a `DataFrame` from a dict of columns and inspect `.dtypes` (noting the pandas 3.x `str` dtype).
3. Use the Index for label-based lookup: `set_index`, `.loc[label]`, `reset_index`.
4. Explain and demonstrate **automatic index alignment** during arithmetic.

### Time budget — ~65 min
| Segment | Time |
|---|---|
| Framing & objectives | 5 min |
| **A.** The Series | 14 min |
| **B.** The DataFrame & dtypes | 14 min |
| **C.** The Index: label lookup | 14 min |
| **D.** Automatic alignment | 12 min |
| Wrap-up + stretch | 6 min |

### Files you need (in `data/`)
- `users.csv` — 2000 rows: `user_id, signup_date, plan, region, seats` (used in Part C).


In [ ]:
%pip install --upgrade pandas

In [ ]:
%%python data/generate_lab_data.py 

# Generate lab data

In [ ]:
%pip install --upgrade numpy

from pathlib import Path
DATA = Path("data")

import pandas as pd
import numpy as np

print("pandas", pd.__version__)   # target: pandas 3.x on Python 3.13

def check(label, predicate):
    try:
        ok = bool(predicate())
    except Exception as exc:
        ok = False
        label = f"{label}  (raised {type(exc).__name__}: {exc})"
    print(("PASS " if ok else "FAIL "), label)
    return ok

print("ready.")

## Part A — The Series  *(guided)*

A `Series` is a 1-D array with an **index** (labels), a **dtype**, and an
optional **name**. Building one from a list gives a default integer index; from a
dict, the keys *become* the index.


In [ ]:
# From a list -> default RangeIndex(0, 3)
temps = pd.Series([98.6, 101.2, 99.0], name="temp_f")
print(temps)
print("dtype:", temps.dtype, "| name:", temps.name)

# From a dict -> keys become the index labels
scores = pd.Series({"alice": 92, "bob": 87, "carol": 95})
print(scores["alice"], "| index:", list(scores.index))

### Exercise A1 — Build two Series
Create `latency` from the list `[120.5, 98.2, 210.0]` with `name="latency_ms"`,
and `plan_counts` from the dict `{"free": 1203, "pro": 549, "enterprise": 228}`.
Then read `plan_counts["pro"]`.


💡 **Hint.** `pd.Series(data, name=...)` for the list; `pd.Series(mapping)` for
the dict. The dict keys become the labels, so `plan_counts["pro"]` works by label.


In [ ]:
latency = None       # TODO: Series from the list, name="latency_ms"
plan_counts = None    # TODO: Series from the dict
pro_count = None      # TODO: plan_counts["pro"]

In [ ]:
check("A1: latency is a Series named 'latency_ms'",
      lambda: isinstance(latency, pd.Series) and latency.name == "latency_ms")
check("A1: latency has 3 float values",
      lambda: latency.shape == (3,) and latency.dtype == "float64")
check("A1: plan_counts is indexed by plan name",
      lambda: list(plan_counts.index) == ["free", "pro", "enterprise"])
check("A1: pro_count == 549", lambda: int(pro_count) == 549)

## Part B — The DataFrame & dtypes

A `DataFrame` is a dict of columns, each a Series, all sharing one Index. The big
pandas 3.x change to notice: **string columns get a dedicated `str` dtype
automatically** — they are no longer the old catch-all `object`.


In [ ]:
data = {
    "name":  ["alice", "bob", "carol"],
    "score": [92, 87, 95],
    "grade": ["A", "B", "A"],
}
df = pd.DataFrame(data)
print(df)
print(df.dtypes)   # name -> str (pandas 3.x!), score -> int64, grade -> str

### Exercise B1 — Build a DataFrame and read its dtypes
Build `models_df` from these three columns:
`model = ["atlas-mini", "atlas-pro", "nova-4"]`,
`calls = [3194, 1986, 1633]`,
`avg_latency = [180.4, 402.1, 351.7]`.
Then capture the dtype of the `model` column as a string in `model_dtype`.


In [ ]:
models_df = None      # TODO: DataFrame from the three columns
model_dtype = None    # TODO: str(models_df["model"].dtype)

In [ ]:
check("B1: models_df is 3x3", lambda: models_df.shape == (3, 3))
check("B1: model column is the pandas 3.x str dtype",
      lambda: model_dtype == "str")
check("B1: calls is an integer column",
      lambda: str(models_df["calls"].dtype) == "int64")
check("B1: avg_latency is a float column",
      lambda: str(models_df["avg_latency"].dtype) == "float64")

## Part C — The Index: label-based lookup

The Index is a *labeled axis*, not just row numbers. `set_index` promotes a column
to the row labels so you can look rows up by meaningful keys with `.loc`;
`reset_index` turns the labels back into a regular column.


In [ ]:
users = pd.read_csv(DATA / "users.csv", parse_dates=["signup_date"])
print("default index:", users.index[:3].tolist(), "...")
print("shape:", users.shape)
users.head(3)

### Exercise C1 — Index by user_id, then look one up
Set the index of `users` to `user_id` (call the result `by_id`), then use `.loc`
to pull the single row for user `10000` into `first_user`. Read its `plan` into
`first_plan`.


💡 **Hint.** `users.set_index("user_id")` returns a new frame whose row labels are
the user ids. Then `by_id.loc[10000]` returns that user's row as a Series, and
`by_id.loc[10000, "plan"]` (or `first_user["plan"]`) gives one field.


In [ ]:
by_id = None         # TODO: users indexed by user_id
first_user = None    # TODO: by_id.loc[10000]  (a Series: that user's row)
first_plan = None    # TODO: that user's plan value

In [ ]:
check("C1: by_id is indexed by user_id",
      lambda: by_id.index.name == "user_id")
check("C1: user_id is no longer a column",
      lambda: "user_id" not in by_id.columns)
check("C1: first_user is user 10000's row",
      lambda: isinstance(first_user, pd.Series) and first_user["seats"] == by_id.loc[10000, "seats"])
check("C1: first_plan matches the frame",
      lambda: (first_plan == by_id.loc[10000, "plan"]) or (pd.isna(first_plan) and pd.isna(by_id.loc[10000, "plan"])))

### Exercise C2 — Restore the integer index
Call `reset_index()` on `by_id` to get `restored`, moving `user_id` back to a
regular column. Confirm `user_id` is a column again and the row count is unchanged.


In [ ]:
restored = None      # TODO: by_id.reset_index()

In [ ]:
check("C2: user_id is a column again", lambda: "user_id" in restored.columns)
check("C2: default RangeIndex restored", lambda: isinstance(restored.index, pd.RangeIndex))
check("C2: no rows lost", lambda: len(restored) == len(users))

## Part D — Automatic index alignment  *(the superpower)*

When you combine two Series, pandas aligns them **by index label**, not by
position. Matching labels are paired; labels present in only one side produce
`NaN`. This is what makes joins and arithmetic "just work" — and what surprises
people coming from NumPy, where alignment is purely positional.


In [ ]:
q1 = pd.Series({"atlas-mini": 100, "atlas-pro": 50, "nova-4": 30})
q2 = pd.Series({"atlas-pro": 60, "nova-4": 40, "orion-8b": 20})
print(q1 + q2)   # aligned by label; unmatched labels -> NaN

### Exercise D1 — Align two token totals
Two Series give token totals by model for two weeks. Add them into `combined`.
Because `orion-8b` appears only in `week2` and `atlas-mini` only in `week1`, those
two labels should come back as `NaN`. Count the non-null results into `n_matched`.


💡 **Hint.** `week1 + week2` aligns by label automatically. `combined.notna().sum()`
counts the labels that appeared in *both*.


In [ ]:
week1 = pd.Series({"atlas-mini": 1200, "atlas-pro": 800, "nova-4": 600})
week2 = pd.Series({"atlas-pro": 900, "nova-4": 500, "orion-8b": 300})
combined = None      # TODO: week1 + week2 (label-aligned)
n_matched = None     # TODO: how many labels are non-null?

In [ ]:
check("D1: atlas-pro summed correctly (800+900)",
      lambda: combined["atlas-pro"] == 1700)
check("D1: orion-8b is NaN (only in week2)",
      lambda: pd.isna(combined["orion-8b"]))
check("D1: atlas-mini is NaN (only in week1)",
      lambda: pd.isna(combined["atlas-mini"]))
check("D1: exactly 2 labels matched both weeks",
      lambda: n_matched == 2)

## Stretch goals *(for fast finishers)*

**S1 — `fill_value` on alignment.** Redo the D1 addition with
`week1.add(week2, fill_value=0)` into `combined_filled` so unmatched labels keep
their single-week value instead of becoming `NaN`. All four models should be
present with no NaN.

**S2 — Series from a DataFrame column.** From `users`, pull the `seats` column as
a Series `seats_s` and confirm it *is* a `Series` whose `.name` is `"seats"`.


In [ ]:
# S1
combined_filled = None   # TODO: week1.add(week2, fill_value=0)

# S2
seats_s = None           # TODO: the seats column of users

In [ ]:
check("S1: no NaN after fill_value=0",
      lambda: bool(combined_filled.notna().all()) and len(combined_filled) == 4)
check("S1: atlas-mini kept its week1 value",
      lambda: combined_filled["atlas-mini"] == 1200)
check("S2: seats_s is a Series named 'seats'",
      lambda: isinstance(seats_s, pd.Series) and seats_s.name == "seats")

## Wrap-up — what you can now do

- Build a `Series` from a list or dict and read its `dtype`, `index`, and `name`.
- Build a `DataFrame` and read `.dtypes` — including the pandas 3.x automatic `str` dtype.
- Use the Index for label lookup with `set_index` / `.loc` / `reset_index`.
- Explain and use automatic **index alignment** in arithmetic (and `fill_value` to control it).

**Next:** Lab 2 — reading and writing data across CSV, Parquet, and JSON, and why
Parquet is the right choice for pipeline intermediates.
